In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp

spark = SparkSession.builder.getOrCreate()

In [ ]:
df_category = spark.read.parquet('../data/bronze/category')
df_customers = spark.read.parquet('../data/bronze/customers')
df_geolocation = spark.read.parquet('../data/bronze/geolocation')
df_order_items = spark.read.parquet('../data/bronze/order_items')
df_order_payments = spark.read.parquet('../data/bronze/order_payments')
df_order_reviews = spark.read.parquet('../data/bronze/order_reviews')
df_orders = spark.read.parquet('../data/bronze/orders')
df_products = spark.read.parquet('../data/bronze/products')
df_sellers = spark.read.parquet('../data/bronze/sellers')

In [ ]:
df_category.printSchema()
df_category = df_category.dropDuplicates()
df_category.count()

In [ ]:
df_customers.printSchema()
df_customers = df_customers.dropna(subset=["customer_id"])
df_customers = df_customers.dropDuplicates(["customer_id"])
df_customers.count()

In [ ]:
from pyspark.sql.functions import avg, col

df_geolocation = spark.read.parquet('../data/silver/geolocation')

df_geolocation = df_geolocation.withColumn("geolocation_lat", col("geolocation_lat").cast("double"))
df_geolocation = df_geolocation.withColumn("geolocation_lng", col("geolocation_lng").cast("double"))
df_geolocation = df_geolocation.dropna(subset=["geolocation_zip_code_prefix"])

df_geolocation = df_geolocation.groupBy("geolocation_zip_code_prefix", "geolocation_city", "geolocation_state") \
    .agg(
        avg("geolocation_lat").alias("geolocation_lat"),
        avg("geolocation_lng").alias("geolocation_lng")
    )

df_geolocation.printSchema()
df_geolocation.count()

In [ ]:
df_order_items.printSchema()
df_order_items = df_order_items.dropna(subset=["order_id", "product_id", "seller_id"])
df_order_items = df_order_items.dropDuplicates()
df_order_items = df_order_items.withColumn("order_item_id", col("order_item_id").cast("integer"))
df_order_items = df_order_items.withColumn("price", col("price").cast("float"))
df_order_items = df_order_items.withColumn("freight_value", col("freight_value").cast("float"))
df_order_items = df_order_items.withColumn("shipping_limit_date", to_timestamp("shipping_limit_date"))
df_order_items.count()


In [ ]:
df_order_payments.printSchema()
df_order_payments = df_order_payments.dropna(subset=["order_id"])
df_order_payments = df_order_payments.dropDuplicates()
df_order_payments = df_order_payments.withColumn("payment_sequential", col("payment_sequential").cast("integer"))
df_order_payments = df_order_payments.withColumn("payment_installments", col("payment_installments").cast("integer"))
df_order_payments = df_order_payments.withColumn("payment_value", col("payment_value").cast("float"))
df_order_payments.count()

In [ ]:
df_order_payments.groupBy("order_id").count().filter("count > 1").show(10)

In [ ]:
df_order_payments.filter(col("order_id").like("8ca5bdac5ebe8f2d6%")).show()

In [ ]:
df_order_reviews.printSchema()
df_order_reviews = df_order_reviews.dropna(subset=["review_id", "order_id"])
df_order_reviews = df_order_reviews.dropDuplicates(["review_id", "order_id"])
df_order_reviews = df_order_reviews.withColumn("review_score", col("review_score").cast("integer"))
df_order_reviews = df_order_reviews.withColumn("review_creation_date", to_timestamp("review_creation_date"))
df_order_reviews = df_order_reviews.withColumn("review_answer_timestamp", to_timestamp("review_answer_timestamp"))
df_order_reviews = df_order_reviews.dropna(subset=["review_score"])
df_order_reviews.count()

In [ ]:
df_orders.printSchema()
df_orders = df_orders.dropna(subset=["order_id", "customer_id"])
df_orders = df_orders.dropDuplicates(["order_id"])
df_orders = df_orders.withColumn("order_purchase_timestamp", to_timestamp("order_purchase_timestamp"))
df_orders = df_orders.withColumn("order_approved_at", to_timestamp("order_approved_at"))
df_orders = df_orders.withColumn("order_delivered_carrier_date", to_timestamp("order_delivered_carrier_date"))
df_orders = df_orders.withColumn("order_delivered_customer_date", to_timestamp("order_delivered_customer_date"))
df_orders = df_orders.withColumn("order_estimated_delivery_date", to_timestamp("order_estimated_delivery_date"))
df_orders.count()

In [ ]:
df_products.printSchema()
df_products = df_products.dropna(subset=["product_id"])
df_products = df_products.dropDuplicates(["product_id"])
df_products = df_products.withColumn("product_name_lenght", col("product_name_lenght").cast("integer"))
df_products = df_products.withColumn("product_description_lenght", col("product_description_lenght").cast("integer"))
df_products = df_products.withColumn("product_photos_qty", col("product_photos_qty").cast("integer"))
df_products = df_products.withColumn("product_weight_g", col("product_weight_g").cast("integer"))
df_products = df_products.withColumn("product_length_cm", col("product_length_cm").cast("integer"))
df_products = df_products.withColumn("product_height_cm", col("product_height_cm").cast("integer"))
df_products = df_products.withColumn("product_width_cm", col("product_width_cm").cast("integer"))
df_products.count()


In [ ]:
df_sellers.printSchema()
df_sellers = df_sellers.dropna(subset=["seller_id"])
df_sellers = df_sellers.dropDuplicates(["seller_id"])
df_sellers.count()

In [ ]:
df_orders = df_orders.filter(
    (col("order_delivered_customer_date") >= col("order_purchase_timestamp")) |
    col("order_delivered_customer_date").isNull()
)

df_orders = df_orders.filter(
    (col("order_delivered_customer_date") >= col("order_delivered_carrier_date")) |
    col("order_delivered_customer_date").isNull()
)

df_orders = df_orders.filter(
    (col("order_approved_at") >= col("order_purchase_timestamp")) |
    col("order_approved_at").isNull()
)

In [ ]:
df_orders.write.parquet('../data/silver/orders', mode='overwrite')
df_order_items.write.parquet('../data/silver/order_items', mode='overwrite')
df_order_payments.write.parquet('../data/silver/order_payments', mode='overwrite')
df_order_reviews.write.parquet('../data/silver/order_reviews', mode='overwrite')
df_customers.write.parquet('../data/silver/customers', mode='overwrite')
df_products.write.parquet('../data/silver/products', mode='overwrite')
df_sellers.write.parquet('../data/silver/sellers', mode='overwrite')
df_category.write.parquet('../data/silver/category', mode='overwrite')
df_geolocation.write.parquet('../data/silver/geolocation', mode='overwrite')